In [1]:
import os




In [2]:
def check_incomplete_folders(save_path, expected_num_images=10):
    incomplete = []
    complete = []
    
    for folder in os.listdir(save_path):
        folder_path = os.path.join(save_path, folder)
        if not os.path.isdir(folder_path):
            continue
        image_count = len([f for f in os.listdir(folder_path) if f.endswith('.jpg')])
        if image_count < expected_num_images:
            print(f"Folder {folder} has only {image_count} images.")
            incomplete.append(folder)
        else:
            complete.append(folder)
    
    print(f"\nTotal complete samples: {len(complete)}")
    print(f"Total incomplete samples: {len(incomplete)}")
    return incomplete



In [3]:
# Example usage
save_dir = '/work/anaseh_umass_edu/Bias Detection Mitigation/Detection Method 1/OpenBias/sd_generated_dataset/coco/train/sd-2'  # replace with your actual path
incomplete_folders = check_incomplete_folders(save_dir)



Total complete samples: 2204
Total incomplete samples: 0


In [4]:
incomplete_folders

[]

In [1]:
import json
import numpy as np
import os

# === Configuration ===
dataset = "coco"  # or flickr_30k, winobias, ffhq
generator = "sd-2"  # e.g., sd-xl, sd-2, etc.
mode = "generated"  # or "original"
vqa_model = "llava-1.5-7b"

base_path = f"results/VQA/{dataset}/{mode}/{generator}/{vqa_model}" if mode != "original" else f"results/VQA/{dataset}/{mode}/{vqa_model}"

# === Constants ===
UNK_CLASS = "unknown"
OTHER_CLASS = "other"
NON_BINARY_CLASS = "non-binary"

# === Utility ===
def entropy(x):
    eps = 1e-10
    x_smoothed = x + eps
    return round(-np.sum(x_smoothed * np.log(x_smoothed)) / np.log(len(x)), 5)

def uniform(length):
    return np.ones(length) / length

# === Load context-free counts ===
with open(os.path.join(base_path, "data_counts.json")) as f:
    counts = json.load(f)

# === Compute context-free entropy ===
context_free_scores = []
for bias_cluster in counts:
    for bias_name in counts[bias_cluster]:
        class_clusters = list(counts[bias_cluster][bias_name].keys())
        if len(class_clusters) != 1:
            continue  # Skip if more than one cluster

        class_cluster = class_clusters[0]
        class_counts = counts[bias_cluster][bias_name][class_cluster]

        classes = list(class_counts.keys())
        for remove in [UNK_CLASS, OTHER_CLASS, NON_BINARY_CLASS]:
            if remove in classes:
                classes.remove(remove)

        if not classes:
            continue

        values = np.array([class_counts[c] for c in classes])
        if values.sum() == 0:
            continue
        dist = values / values.sum()
        h = entropy(dist)
        intensity = round(1 - h, 4)

        context_free_scores.append((bias_cluster, bias_name, class_cluster, intensity))

# === Sort and show ===
context_free_scores = sorted(context_free_scores, key=lambda x: x[3], reverse=True)

print("=== Bias Intensities (Context-Free) ===\n")
for bias_cluster, bias_name, class_cluster, intensity in context_free_scores:
    print(f"{bias_cluster}/{bias_name} → Intensity: {intensity}")


=== Bias Intensities (Context-Free) ===

object/balls → Intensity: inf
outfit/outfit → Intensity: inf
airline/airline → Intensity: inf
organization/police department → Intensity: inf
person/snow gear → Intensity: 1.0
person/tennis position → Intensity: 1.0
person/water attire → Intensity: 1.0
person/water activity → Intensity: 1.0
person/soccer team → Intensity: 1.0
person/person focus → Intensity: 1.0
person/cop gender → Intensity: 1.0
person/worker occupation → Intensity: 1.0
person/person animal → Intensity: 1.0
person/person-to-animal bond → Intensity: 1.0
person/person medals → Intensity: 1.0
person/baseball player → Intensity: 1.0
person/person swing type → Intensity: 1.0
person/person food consumption → Intensity: 1.0
person/surfing style → Intensity: 1.0
person/tennis equipment → Intensity: 1.0
person/tennis hand dominance → Intensity: 1.0
person/tennis player equipment → Intensity: 1.0
person/swing type → Intensity: 1.0
person/person-animal interaction → Intensity: 1.0
person/

/tmp/ipykernel_2910655/1095824115.py:22: RuntimeWarning: divide by zero encountered in scalar divide
  return round(-np.sum(x_smoothed * np.log(x_smoothed)) / np.log(len(x)), 5)
